# CS 521 HW3, Problem 2: SynCode

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

SynCode with Microsoft's Phi-2 extracts the two operands and the operator from the query.
The grammar forces the output to be `<number> <operator> <number>`, and Python calculates the answer.

In [ ]:
!pip install -q syncode accelerate

In [1]:
import operator

from syncode import Syncode

grammar = r"""
start: " " NUMBER " " OP " " NUMBER
OP: "+" | "-" | "*" | "/"
NUMBER: /-?[0-9]+(\.[0-9]*)?/
"""

few_shot = """Extract the two numbers and the operator from the question.

Question: What is 12.5 plus 3.0?
Answer: 12.5 + 3.0

Question: What is 100. minus 45.25?
Answer: 100. - 45.25

Question: What is 7.2 times 3.?
Answer: 7.2 * 3.

Question: What is 81.0 divided by 9.0?
Answer: 81.0 / 9.0

Question: {query}
Answer:"""

syn_llm = Syncode(model="microsoft/phi-2", grammar=grammar, mode="grammar_strict", parse_output_only=True, max_new_tokens=20)

[2026-10-27 20:17:20,257-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


[2026-10-27 20:18:16,582-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


[2026-10-27 20:18:30,985-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_7595064079_50257.pkl exist: False


[2026-10-27 20:18:30,986-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /tmp/claude-1307560/-home-db50/423ed581-b9fc-444c-8d7c-86f505857b74/scratchpad/run2/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_7595064079_50257.pkl.


[2026-10-27 20:18:31,378-syncode.mask_store.fsm_set] - 3 FSMs with 13 states initialized in 0.00 seconds


[2026-10-27 20:18:31,379-syncode.mask_store.mask_store] - Ignore whitespace is False


[2026-10-27 20:18:31,388-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 9 to 4


[2026-10-27 20:18:33,941-syncode.mask_store.mask_store] - Time taken to create mask store: 2.95 seconds


In [2]:
OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}


def run_syncode(textual_query):
    output = syn_llm.infer(few_shot.format(query=textual_query))[0]
    a, op, b = output.split()
    operands = [float(a), float(b)]
    return operands, op, OPS[op](*operands)

In [3]:
for query in ["What is 327. multiplied by 11.0?", "What is 45.1 plus 23.54?", "What is 120.4 divided by 4.0?"]:
    print(query, run_syncode(query))

What is 327. multiplied by 11.0? ([327.0, 11.0], '*', 3597.0)
What is 45.1 plus 23.54? ([45.1, 23.54], '+', 68.64)


What is 120.4 divided by 4.0? ([120.4, 4.0], '/', 30.1)


# Additional experiments

In [4]:
extra_queries = [
    "What is 9.5 minus 2.25?",
    "What is 3.3 times 2.?",
    "What is 1000. divided by 8.0?",
    "What is the sum of 2.5 and 4.75?",
    "What is 64.0 subtracted by 16.5?",
    "What is 0.5 multiplied by 0.25?",
    "What is 17. plus 0.003?",
    "What is the product of 2.0 and 3.5?",
    "What is 12.0 over 4.0?",
    "What is -3.5 plus 2.0?",
    "What is 5.5 added to 4.5?",
    "What is the quotient of 9.0 and 3.0?",
    "What is 2.75 less 1.25?",
    "What is 123.456 multiplied by 7.89?",
    "What is 10.0 divided by 3.0?",
]
for query in extra_queries:
    print(query, run_syncode(query))

What is 9.5 minus 2.25? ([9.5, 2.25], '-', 7.25)
What is 3.3 times 2.? ([3.3, 2.0], '*', 6.6)
What is 1000. divided by 8.0? ([1000.0, 8.0], '/', 125.0)
What is the sum of 2.5 and 4.75? ([2.5, 4.75], '+', 7.25)
What is 64.0 subtracted by 16.5? ([64.0, 16.5], '-', 47.5)
What is 0.5 multiplied by 0.25? ([0.5, 0.25], '*', 0.125)
What is 17. plus 0.003? ([17.0, 0.003], '+', 17.003)
What is the product of 2.0 and 3.5? ([2.0, 3.5], '*', 7.0)
What is 12.0 over 4.0? ([12.0, 4.0], '/', 3.0)
What is -3.5 plus 2.0? ([-3.5, 2.0], '+', -1.5)
What is 5.5 added to 4.5? ([5.5, 4.5], '+', 10.0)
What is the quotient of 9.0 and 3.0? ([9.0, 3.0], '/', 3.0)
What is 2.75 less 1.25? ([2.75, 1.25], '-', 1.5)
What is 123.456 multiplied by 7.89? ([123.456, 7.89], '*', 974.0678399999999)
What is 10.0 divided by 3.0? ([10.0, 3.0], '/', 3.3333333333333335)
